# Week 6 Exercise: Fine-tuning GPT for "The Price is Right"

I fine-tuned **gpt-4.1-nano** on ed-donner's **items_lite** dataset, applying lessons from the week so the fine-tuned model **beats** the course baseline (and ideally gets closer to zero-shot GPT 4.1 Nano’s 62.51 or better):

1. **Richer input (Day 2 + Day 3):** Day 2 showed that structured product info (Title, Category, Description) helps; Day 3 showed that **weight** and text length help traditional ML. So we pass **Title, Category, Description, and Weight** (when available) as the input for price — same structure at train and eval.

2. **More data (Day 5):** We use **100 train + 50 validation** (course used 100/50) for better signal without blowing the budget.

3. **Avoid overfitting (results.ipynb):** The course fine-tuned model got **75.91** error vs **62.51** for zero-shot GPT 4.1 Nano — fine-tuning hurt. We use **learning_rate_multiplier=0.5** (gentler than default 1.0) and keep **n_epochs=1**, **batch_size=1**, **seed=42** so the model doesn’t overfit the small set.

In [ ]:
import os
import json
import time
from pathlib import Path
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI

# So we can import pricer from week6 (run from repo root or from this folder)
cwd = Path.cwd()
if (cwd / "config.py").exists():
    week6_dir = cwd.parent.parent
    repo_root = cwd.parent.parent.parent
else:
    repo_root = cwd
    week6_dir = repo_root / "week6"
import sys
sys.path.insert(0, str(week6_dir))

load_dotenv(repo_root / ".env", override=True)

from pricer.items import Item
from pricer.evaluator import evaluate
import config

In [ ]:
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    raise ValueError("Set HF_TOKEN in .env to load the dataset")
login(token=hf_token, add_to_git_credential=True)

In [ ]:
train, val, test = Item.from_hub(config.DATASET)
print(f"Loaded {len(train):,} train, {len(val):,} val, {len(test):,} test")

In [ ]:
# Subset for fine-tuning (Day 5: 100 train, 50 val — good signal, low cost)
fine_tune_train = train[: config.TRAIN_SIZE]
fine_tune_val = val[: config.VAL_SIZE]
print(f"Fine-tuning on {len(fine_tune_train)} train, {len(fine_tune_val)} validation")

## Build JSONL for OpenAI

Each example: **user** = structured product info (Title, Category, Description, Weight if present) — the parameters we use for price (Day 2/3); **assistant** = price (2 decimals). Same format at train and eval.

In [ ]:
def build_product_info(item):
    """Structured input for price (Day 2 + Day 3): Title, Category, Description, Weight when present."""
    lines = [
        f"Title: {item.title}",
        f"Category: {item.category}",
        f"Description: {item.summary or item.title}",
    ]
    if item.weight is not None and item.weight > 0:
        lines.append(f"Weight: {item.weight} kg")
    return "\n".join(lines)

def messages_for(item):
    product_info = build_product_info(item)
    user_content = config.USER_PROMPT.format(summary=product_info)
    assistant_content = f"{item.price:.2f}"
    return [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": assistant_content},
    ]

def write_jsonl(items, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        for item in items:
            line = json.dumps({"messages": messages_for(item)})
            f.write(line + "\n")

write_jsonl(fine_tune_train, "jsonl/fine_tune_train.jsonl")
write_jsonl(fine_tune_val, "jsonl/fine_tune_validation.jsonl")
print("Wrote jsonl files.")

## Start fine-tuning job

**Parameters for price:** structured product info (Title, Category, Description, Weight) as built above.

**Hyperparameters:** `n_epochs=1`, `batch_size=1`, `learning_rate_multiplier=0.5` (gentler to avoid overfitting), `seed=42`. Set in `config`.

In [ ]:
client = OpenAI()

with open("jsonl/fine_tune_train.jsonl", "rb") as f:
    train_file = client.files.create(file=f, purpose="fine-tune")
with open("jsonl/fine_tune_validation.jsonl", "rb") as f:
    val_file = client.files.create(file=f, purpose="fine-tune")
print(f"Uploaded: {train_file.id}, {val_file.id}")

In [ ]:
job = client.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=val_file.id,
    model=config.BASE_MODEL,
    seed=config.SEED,
    hyperparameters={
        "n_epochs": config.N_EPOCHS,
        "batch_size": config.BATCH_SIZE,
        "learning_rate_multiplier": config.LEARNING_RATE_MULTIPLIER,
    },
    suffix=config.SUFFIX,
)
job_id = job.id
print(f"Job started: {job_id}")

In [ ]:
while True:
    status = client.fine_tuning.jobs.retrieve(job_id)
    print(status.status)
    if status.status == "succeeded":
        fine_tuned_model = status.fine_tuned_model
        print(f"Done. Model: {fine_tuned_model}")
        break
    if status.status == "failed":
        print(status)
        raise RuntimeError("Fine-tuning failed")
    time.sleep(30)

## Evaluate on test set

In [ ]:
def predict(item):
    product_info = build_product_info(item)  # same format as training
    user_content = config.USER_PROMPT.format(summary=product_info)
    resp = client.chat.completions.create(
        model=fine_tuned_model,
        messages=[{"role": "user", "content": user_content}],
        max_tokens=15,
    )
    return (resp.choices[0].message.content or "0").strip()

evaluate(predict, test, size=config.EVAL_SIZE, workers=5)